# EventFlow — Baseline Template

This notebook provides a minimal end-to-end baseline for **EventFlow: Temporal Event Prediction**.

The task is:

> Given the observed history of a system, predict a probability distribution over the **next event**.

The primary evaluation metric is **multiclass Log Loss**:

$$
\mathrm{LogLoss}
=
-\frac{1}{N}
\sum_{i=1}^{N}
\log p_{i,y_i}.
$$

**Lower is better.**

The baseline deliberately uses a simple **multinomial logistic regression** and only the **current event row**. It is intended as a starting point, not a competitive solution.

The workflow is:

**load data → construct next-event targets → local validation → refit on all training data → evaluate on the provided test set**

Use your own train/validation splits for model development. The provided test set is a common final evaluation set and should not be added to the training data.


## 1. Imports and data location

The release contains exactly three CSV files:

- `eventflow_train.csv`
- `eventflow_test.csv`
- `eventflow_test_labels.csv`

If needed, set `DATA_DIR` to the folder containing them.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


RANDOM_STATE = 2210

# If needed, replace None with the folder containing the EventFlow CSV files.
DATA_DIR = None


def find_data_dir():
    candidates = [Path("."), Path("data"), Path("../data")]

    required = {
        "eventflow_train.csv",
        "eventflow_test.csv",
        "eventflow_test_labels.csv",
    }

    for candidate in candidates:
        if all((candidate / name).exists() for name in required):
            return candidate

    raise FileNotFoundError(
        "Could not locate the EventFlow CSV files automatically. "
        "Set DATA_DIR above to the directory containing the three required files."
    )


if DATA_DIR is None:
    DATA_DIR = find_data_dir()
else:
    DATA_DIR = Path(DATA_DIR)

print("Using data from:", DATA_DIR)


## 2. Load and inspect the data

Each `sequence_id` identifies one independent system history. Rows within each sequence are chronological.

- `eventflow_train.csv` contains complete training histories.
- `eventflow_test.csv` contains held-out prefixes ending immediately before the event to predict.
- `eventflow_test_labels.csv` contains the true next event for each test sequence.


In [ ]:
train = pd.read_csv(DATA_DIR / "eventflow_train.csv")
test = pd.read_csv(DATA_DIR / "eventflow_test.csv")
test_labels = pd.read_csv(DATA_DIR / "eventflow_test_labels.csv")

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")

print(f"\nTraining sequences: {train['sequence_id'].nunique():,}")
print(f"Test sequences:     {test['sequence_id'].nunique():,}")

assert test_labels["sequence_id"].is_unique
assert set(test_labels["sequence_id"]) == set(test["sequence_id"].unique())

print("\nTraining data:")
display(train.head())

print("\nTest labels:")
display(test_labels.head())


In [ ]:
# A few basic checks. These are only a starting point for your own analysis.

sequence_lengths = train.groupby("sequence_id").size()

print("Training sequence lengths:")
display(sequence_lengths.describe())

print("\nEvent frequencies:")
display(
    train["event_type"]
    .value_counts(normalize=True)
    .rename("fraction")
    .to_frame()
)

missing = train.isna().mean().sort_values(ascending=False)
print("\nColumns with missing values:")
display(missing[missing > 0].rename("missing_fraction").to_frame())


## 3. Construct next-event training examples

Within each training sequence, the target for a row is the `event_type` on the following row. The last event of each training sequence has no observed next event, so it is excluded from supervised training.


In [ ]:
data = train.copy()

data["target"] = (
    data.groupby("sequence_id")["event_type"].shift(-1)
)

data = data.dropna(subset=["target"]).reset_index(drop=True)

print(f"Supervised training examples: {len(data):,}")
display(
    data[["sequence_id", "event_index", "event_type", "target"]].head(10)
)


## 4. Simple local validation split

Keep complete systems together: a `sequence_id` used for fitting should not also appear in validation.

This split is intentionally simple. You should decide what validation strategy is appropriate for your own experiments.


In [ ]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_idx, val_idx = next(
    splitter.split(data, groups=data["sequence_id"])
)

train_local = data.iloc[train_idx].copy()
val_local = data.iloc[val_idx].copy()

print(f"Local training examples:   {len(train_local):,}")
print(f"Local validation examples: {len(val_local):,}")
print(f"Training sequences:        {train_local['sequence_id'].nunique():,}")
print(f"Validation sequences:      {val_local['sequence_id'].nunique():,}")


## 5. Current-row logistic-regression baseline

This baseline uses only information from the **current row**. It does not construct historical summaries or explicitly model the event sequence.


In [ ]:
categorical_cols = [
    col for col in [
        "event_type",
        "subsystem",
        "shift_code",
        "operator_group",
        "device_family",
        "firmware_channel",
    ]
    if col in data.columns
]

# Use raw numeric measurements, but deliberately exclude identifiers
# and absolute event position/time from this simple baseline.
excluded_cols = {
    "sequence_id",
    "event_index",
    "timestamp",
    "target",
} | set(categorical_cols)

numeric_cols = [
    col for col in data.columns
    if col not in excluded_cols
    and pd.api.types.is_numeric_dtype(data[col])
]

feature_cols = categorical_cols + numeric_cols

print("Categorical features:")
print(categorical_cols)

print("\nNumeric features:")
print(numeric_cols)


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_cols,
        ),
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_cols,
        ),
    ]
)

baseline_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=400)),
])

baseline_model.fit(
    train_local[feature_cols],
    train_local["target"],
)

val_prob = baseline_model.predict_proba(
    val_local[feature_cols]
)

val_logloss = log_loss(
    val_local["target"],
    val_prob,
    labels=baseline_model.named_steps["model"].classes_,
)

print(f"Local validation Log Loss: {val_logloss:.4f}")


### Interpreting the baseline

The score above is a local validation result. It gives you a basic reference point and checks that the prediction pipeline is working.

The project is about going beyond this current-row baseline by investigating useful representations of the history, comparing modeling approaches, and supporting design choices with controlled experiments.


## 6. Refit on all training examples

After completing local development for this baseline, refit it using all available supervised training examples before evaluating on the common test set.


In [ ]:
baseline_model.fit(
    data[feature_cols],
    data["target"],
)

print("Baseline refit on all available training examples.")


## 7. Evaluate on the provided test set

Each test sequence stops immediately before the event to be predicted. We therefore take the **last observed row** of each test sequence, apply the baseline model, and compare the predicted probability distribution with `eventflow_test_labels.csv`.

The test labels are provided for common final evaluation. Do not add these examples to the training set.


In [ ]:
test_last = (
    test.sort_values(["sequence_id", "event_index"])
    .groupby("sequence_id", as_index=False)
    .tail(1)
    .set_index("sequence_id")
)

# Match the test-row order to the label file.
test_last = test_last.loc[test_labels["sequence_id"]]

test_prob = baseline_model.predict_proba(
    test_last[feature_cols]
)

model_classes = baseline_model.named_steps["model"].classes_

test_logloss = log_loss(
    test_labels["next_event"],
    test_prob,
    labels=model_classes,
)

print(f"Test Log Loss: {test_logloss:.4f}")


## Where to go from here

This notebook is intentionally a **starting point, not a competitive solution**.

For the project, go beyond the current-row baseline. Among other things, consider:

- what aspects of the history may contain predictive information;
- which raw variables are useful, redundant, noisy, or missing;
- how to represent information from the event history;
- how different modeling approaches make use of sequential information;
- how to tune models and compare them fairly;
- which controlled ablations help explain why a method works or fails;
- how to produce and evaluate multi-step event-sequence generations.

Keep a clear record of your experiments and use your local validation procedure to make model-development decisions.
